# Trigger-selection sensitivity on the Road Traffic Fine Management log

Episode segmentation (method.tex, Definition *episode*) hinges on one hyperparameter: the
**trigger activity**. The conformance notebook uses `Create Fine`, which occurs exactly once per
case, so segmentation yields one episode per fine. Choosing a different trigger changes everything
downstream, before a single constraint is evaluated:

* **coverage** — cases that never record the trigger produce zero episodes and drop out of the
  analysis entirely;
* **granularity** — a repeated trigger (e.g. `Payment`) splits a case into several episodes;
* **prefix loss** — events before the first trigger fall outside every episode window and are
  never checked;
* **label validity** — the outcome labeler reads the amount owed from the episode's own
  `Create Fine` event; episodes that lack one get `due = 0 + expenses` and are biased toward
  "Success".

This notebook quantifies those effects descriptively (no conformance checking, no statistics) by
running `build_trigger_episodes` with each candidate trigger and aggregating segmentation metrics
via `compute_segmentation_metrics` (`evaluation/rtfm_utils.py`). The point is to frame trigger
selection as a small hyperparameter-optimization problem an analyst must solve when instantiating
the method in a new domain.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "evaluation" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from evaluation.rtfm_utils import compute_segmentation_metrics

XES_PATH = REPO_ROOT / "evaluation" / "Road_Traffic_Fine_Management_Process.xes"
DATA_DIR = REPO_ROOT / "evaluation" / "data"
FIG_DIR = REPO_ROOT / "evaluation" / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

PARQUET_CACHE = DATA_DIR / "rtfm_event_log.parquet"
METRICS_CSV = DATA_DIR / "trigger_sensitivity.csv"

# Candidate triggers, ordered from process start to escalation. `Create Fine` is
# the baseline used by rtfm_conformance.ipynb; `Appeal to Judge` is deliberately
# rare, to show the coverage-collapse extreme.
TRIGGER_CANDIDATES = [
    "Create Fine",
    "Send Fine",
    "Insert Fine Notification",
    "Add penalty",
    "Payment",
    "Appeal to Judge",
]

# Number of cases to measure. The metric loop is pure Python over per-case
# frames (~30 s per 5,000 cases per trigger); 20,000 keeps the whole sweep
# around 10-15 minutes. Set to None for the full 150,370 cases.
SAMPLE_N = 20_000
RANDOM_STATE = 42

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

## 1. Load the log and pick the working set

Same parquet cache and sampling convention as `rtfm_conformance.ipynb`. The candidate table below
shows, per activity, the share of cases that contain it at least once — an upper bound on the case
coverage any trigger can reach.

In [ ]:
ATTRIBUTE_COLUMNS = [
    "amount",
    "expense",
    "paymentAmount",
    "totalPaymentAmount",
    "points",
    "article",
    "vehicleClass",
    "dismissal",
    "notificationType",
    "lastSent",
]


def load_rtfm_log(force_reparse: bool = False) -> pd.DataFrame:
    """Return the RTFM log as a wide event-log DataFrame, using the parquet cache."""
    if PARQUET_CACHE.exists() and not force_reparse:
        return pd.read_parquet(PARQUET_CACHE)

    import pm4py

    raw = pm4py.read_xes(str(XES_PATH))
    df = raw.rename(
        columns={
            "case:concept:name": "case_id",
            "concept:name": "activity",
            "time:timestamp": "timestamp",
        }
    )
    keep = ["case_id", "activity", "timestamp"] + [c for c in ATTRIBUTE_COLUMNS if c in df.columns]
    df = df[keep].copy()

    df["case_id"] = df["case_id"].astype(str)
    df["activity"] = df["activity"].astype(str)
    ts = pd.to_datetime(df["timestamp"], utc=True)
    df["timestamp"] = ts.dt.tz_localize(None)

    df = df.sort_values(["case_id", "timestamp"], kind="mergesort").reset_index(drop=True)
    df.to_parquet(PARQUET_CACHE, index=False)
    return df


log = load_rtfm_log()
print(f"{len(log):,} events / {log['case_id'].nunique():,} cases")

all_case_ids = log["case_id"].drop_duplicates()
if SAMPLE_N is not None and SAMPLE_N < len(all_case_ids):
    case_ids = set(all_case_ids.sample(n=SAMPLE_N, random_state=RANDOM_STATE))
else:
    case_ids = set(all_case_ids)
sample = log[log["case_id"].isin(case_ids)].copy()
n_cases = sample["case_id"].nunique()
print(f"working set: {n_cases:,} cases / {len(sample):,} events")

# Per-activity case presence: the ceiling on coverage for each candidate.
presence = (
    sample.drop_duplicates(["case_id", "activity"])["activity"]
    .value_counts()
    .to_frame("cases_containing")
)
presence["case_share"] = presence["cases_containing"] / n_cases
presence

## 2. Segmentation metrics per candidate trigger

For each candidate, every case is segmented with `build_trigger_episodes` (fallback-only horizon:
episode runs from a trigger to the next trigger or trace end — identical to the conformance
notebook) and aggregated into one row:

| metric | reads as |
|---|---|
| `case_coverage` | share of cases with ≥ 1 trigger; the rest are **dropped** |
| `eps_per_case_*`, `multi_episode_case_share` | segmentation granularity |
| `ep_events_*`, `ep_duration_days_*` | episode size in events / days |
| `prefix_loss` | share of events (in covered cases) cut before the first trigger |
| `events_lost_to_dropped_cases` | events discarded because their case has no trigger |
| `same_day_collapsed_triggers` | trigger repeats collapsed by the log's day-resolution timestamps |
| `success_share` | per-episode outcome balance under `derive_episode_outcome` |
| `eps_without_create_fine_share` | episodes whose outcome label is computed without the `Create Fine` amount — a **label-validity** warning, not just a size effect |

In [ ]:
rows = []
distributions = {}
for trigger in tqdm(TRIGGER_CANDIDATES, desc="triggers"):
    m = compute_segmentation_metrics(sample, trigger, return_distributions=True)
    distributions[trigger] = {
        "eps_per_case": m.pop("_dist_eps_per_case"),
        "ep_events": m.pop("_dist_ep_events"),
    }
    rows.append(m)

metrics = pd.DataFrame(rows).set_index("trigger")
metrics.to_csv(METRICS_CSV)
print(f"written: {METRICS_CSV}")
metrics.T

In [ ]:
# Sanity checks: the baseline must reproduce the conformance notebook's behaviour.
base = metrics.loc["Create Fine"]
assert base["case_coverage"] == 1.0, "every case starts with Create Fine"
assert base["eps_per_case_max"] == 1, "Create Fine occurs once per case"
assert base["prefix_loss"] == 0.0, "nothing precedes the first event"
assert base["eps_without_create_fine_share"] == 0.0

# Payment is repeated and frequently absent: coverage well below 1, some same-day collapses.
pay = metrics.loc["Payment"]
assert pay["case_coverage"] < 0.7
assert pay["eps_without_create_fine_share"] > 0.5
print("sanity checks passed")

## 3. Figures

Single-hue bars (identity is carried by the axis labels, not by color); the red accent marks the
label-validity warning share.

In [ ]:
import matplotlib.pyplot as plt

BLUE = "#4477AA"
GRAY = "#BBBBBB"
RED = "#EE6677"
order = TRIGGER_CANDIDATES  # fixed order everywhere: process start -> escalation

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def save(fig, name):
    path = FIG_DIR / name
    fig.savefig(path, bbox_inches="tight", dpi=200)
    print(f"saved: {path}")


# -- Case coverage ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 3))
vals = metrics.loc[order, "case_coverage"] * 100
ax.barh(order[::-1], vals[::-1], color=BLUE)
for y, v in enumerate(vals[::-1]):
    ax.text(v + 1, y, f"{v:.1f}%", va="center", fontsize=9)
ax.set_xlim(0, 112)
ax.set_xlabel("cases with ≥ 1 trigger occurrence (%)")
ax.set_title("Case coverage per candidate trigger")
save(fig, "trigger_coverage.png")
plt.show()

In [ ]:
# -- Episodes per case & episode size ----------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.4))

means = metrics.loc[order, "eps_per_case_mean"]
multi = metrics.loc[order, "multi_episode_case_share"] * 100
ax1.bar(range(len(order)), means, color=BLUE)
for x, (m, s) in enumerate(zip(means, multi)):
    ax1.text(x, m + 0.02, f"{m:.2f}\n({s:.0f}% multi)", ha="center", fontsize=8)
ax1.set_xticks(range(len(order)))
ax1.set_xticklabels(order, rotation=30, ha="right", fontsize=8)
ax1.set_ylabel("episodes per covered case (mean)")
ax1.set_title("Granularity")
ax1.set_ylim(0, max(means) * 1.25)

ax2.boxplot(
    [distributions[t]["ep_events"] for t in order],
    tick_labels=order,
    showfliers=False,
    medianprops={"color": BLUE, "linewidth": 2},
)
ax2.set_ylabel("events per episode")
ax2.set_title("Episode size (fliers hidden)")
ax2.tick_params(axis="x", rotation=30, labelsize=8)
for lbl in ax2.get_xticklabels():
    lbl.set_ha("right")

fig.tight_layout()
save(fig, "trigger_granularity.png")
plt.show()

In [ ]:
# -- Data loss: prefix loss + dropped-case loss -------------------------------
# Both series as shares of ALL events in the working set: prefix_loss is stored
# relative to covered-case events, so rescale it before stacking.
fig, ax = plt.subplots(figsize=(7, 3))
covered_events = len(sample) - metrics.loc[order, "events_lost_to_dropped_cases"]
prefix = metrics.loc[order, "prefix_loss"] * covered_events / len(sample) * 100
dropped = (
    metrics.loc[order, "events_lost_to_dropped_cases"] / len(sample) * 100
)
x = np.arange(len(order))
ax.bar(x, dropped, color=GRAY, label="events in dropped cases")
ax.bar(x, prefix, bottom=dropped, color=BLUE, label="prefix loss in covered cases")
for xi, (d, p) in enumerate(zip(dropped, prefix)):
    ax.text(xi, d + p + 1, f"{d + p:.0f}%", ha="center", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(order, rotation=30, ha="right", fontsize=8)
ax.set_ylabel("% of all events never checked")
ax.set_title("Events excluded from every episode")
ax.legend(fontsize=8, frameon=False)
save(fig, "trigger_data_loss.png")
plt.show()

In [ ]:
# -- Outcome balance & label validity -----------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.4))

succ = metrics.loc[order, "success_share"] * 100
x = np.arange(len(order))
ax1.bar(x, succ, color=BLUE, label="Success")
ax1.bar(x, 100 - succ, bottom=succ, color=GRAY, label="Failure")
ax1.set_xticks(x)
ax1.set_xticklabels(order, rotation=30, ha="right", fontsize=8)
ax1.set_ylabel("% of episodes")
ax1.set_title("Episode outcome balance")
ax1.legend(fontsize=8, frameon=False, loc="center left", bbox_to_anchor=(1.0, 0.5))

validity = metrics.loc[order, "eps_without_create_fine_share"] * 100
ax2.bar(x, validity, color=RED)
for xi, v in enumerate(validity):
    ax2.text(xi, v + 2, f"{v:.0f}%", ha="center", fontsize=9)
ax2.set_xticks(x)
ax2.set_xticklabels(order, rotation=30, ha="right", fontsize=8)
ax2.set_ylabel("% of episodes without Create Fine")
ax2.set_ylim(0, 118)
ax2.set_title("Label-validity warning: due amount unavailable")

fig.tight_layout()
save(fig, "trigger_label_validity.png")
plt.show()

## 4. Discussion: trigger selection as hyperparameter optimization

The sweep makes the trade-off space of the episode definition concrete:

* **`Create Fine` is the principled choice for RTFM** — it is universal (100% coverage), initial
  (0% prefix loss) and unique per case, so segmentation is lossless and the outcome labeler always
  sees the amount owed. The price is granularity: one episode per case, so conformance is judged
  once per fine and intra-case repetition (multiple payments, repeated penalties) is never
  isolated.
* **Later or repeated triggers buy granularity with three currencies.** `Payment` splits paying
  cases into per-payment episodes, but (i) drops every never-paid case — which is precisely the
  "Failure"-rich stratum, so the outcome balance shifts and any downstream statistics are computed
  on a biased subpopulation; (ii) cuts the pre-payment prefix, removing the very activations
  (`Create Fine`, `Send Fine`) most rules key on; and (iii) invalidates the outcome label, since
  the due amount lives on an event outside the episode.
* **Rare triggers collapse the analysis.** `Appeal to Judge` keeps only the few appealing cases —
  formally valid segmentation, practically an empty study.
* **Timestamp resolution interacts with the trigger.** RTFM's day-granular timestamps merge
  same-day trigger repeats into one boundary (`same_day_collapsed_triggers`), an effect invisible
  with a once-per-case trigger.

**What to check when instantiating the method in a new domain.** A candidate trigger should be
scored before any conformance run on: case coverage (dropped cases are a selection bias, not just
lost sample size), prefix loss (events before the first trigger are unverifiable), whether the
outcome label's evidence lies inside the episode window, and whether the resulting episode count
per case matches the unit of analysis the research question needs. In the ICU instantiation,
`InvasiveVentilationStarted` plays the same role `Create Fine` plays here: near-universal in the
studied cohort, process-initial, and the natural boundary for the clinical question — the RTFM
sweep shows those properties are not automatic but the result of a (implicit) hyperparameter
choice, and gives the metrics to make that choice explicit.